# Week 7 · Day 1 — Snowflake building blocks: make your own table

*Week 6 you queried tables someone else built. This week you build them — starting with the object hierarchy every Snowflake account is made of.*

**By the end you'll have shipped:** your own **`matters`** table — created with `CREATE TABLE`, populated with `INSERT`, and queried back — plus a clear mental model of **account → warehouse → database → schema → table**.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M4 · Data & Snowflake (Week 7) |
| **Prerequisites** | Week 6 (SELECT / GROUP BY / JOIN) |
| **Est. time** | ~30 min |
| **Capstone slice** | The **store** for *Matter Intelligence* — the table your matters actually live in |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — real Snowflake DDL, run on DuckDB when there's no account |

### 🎯 Learning objectives

By the end you'll be able to:
- Describe Snowflake's **object hierarchy**: account → warehouse → database → schema → table.
- Choose sensible **data types** (`STRING`, `NUMBER`/`DECIMAL`, `DATE`, `BOOLEAN`).
- **`CREATE TABLE`** with columns and types, and **`INSERT`** rows into it.
- Build a table *from a query* with **`CREATE TABLE AS SELECT`** (CTAS).
- Explain why Snowflake **separates storage from compute** (and what a warehouse actually is).

### ⚖️ Why it matters

*Matter Intelligence* has to **store** things — matter metadata, extracted clauses, Claude's summaries. That store is a Snowflake **table**. Before you can load a firm's matters, you need to know how a warehouse is organized and how to define a table that fits the data. Today you build the exact table the rest of the capstone writes to.

### ⚙️ Setup

Just the `run_sql(...)` helper — **no tables preloaded**, because today *you* create them. (On Snowflake this runs against your account; offline it runs on DuckDB, which speaks the same DDL.)

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

### 1 · The Snowflake hierarchy — a legal analogy

Snowflake nests its objects. From biggest to smallest:

| Snowflake object | What it is | Legal analogy |
|---|---|---|
| **Account** | your whole Snowflake instance | the firm's building |
| **Warehouse** | *compute* — the engine that runs queries (billed per-second) | the electricity / the associates doing the work |
| **Database** | a top-level container of data | a filing cabinet |
| **Schema** | a folder of related tables inside a database | a drawer in the cabinet |
| **Table** | rows and columns of one kind of thing | a folder of like documents |

The big idea: a **warehouse is compute, not storage.** Your data sits in databases/schemas; a *warehouse* is the muscle you switch on to query it — and you only pay while it runs. You'd address a table by its full path: `DATABASE.SCHEMA.TABLE` (e.g. `LEGAL.PUBLIC.MATTERS`).

### 2 · Data types — declaring what each column holds

A table's columns each have a **type**. The handful you'll use constantly:

| Type | Holds | Example |
|---|---|---|
| **`STRING`** (a.k.a. `VARCHAR`/`TEXT`) | text | `'Acme Corp'` |
| **`NUMBER(p,s)`** / **`DECIMAL`** | exact numbers (money!) | `18500.00` |
| **`FLOAT`** | approximate decimals | `3.14159` |
| **`DATE`** | a calendar date | `'2026-01-12'` |
| **`BOOLEAN`** | true / false | `TRUE` |

> Use **`NUMBER`/`DECIMAL`** for money (exact), not `FLOAT` (which can round oddly). Snowflake's native money type is `NUMBER(38,2)`; DuckDB spells the same thing `DECIMAL(38,2)` — we'll use `DECIMAL` so the lesson runs on both.

### 3 · `CREATE TABLE` — define the shape

`CREATE TABLE name (column type, ...)` defines an empty table. `CREATE OR REPLACE` makes it re-runnable (drops any old copy first) — handy in a notebook.

In [ ]:
run_sql("""
    CREATE OR REPLACE TABLE matters (
        matter_id      STRING,
        client         STRING,
        practice_area  STRING,
        status         STRING,
        open_date      DATE,
        amount_billed  DECIMAL(12, 2),
        is_privileged  BOOLEAN,
        lead_attorney  STRING
    )
""")
print("✅ empty `matters` table created")
run_sql("SELECT * FROM matters")   # 0 rows, but the columns exist

**What just happened:** you defined a table with eight typed columns and no rows yet. The empty `SELECT *` proves the *shape* exists — a schema waiting for data.

### 4 · `INSERT` — add rows

`INSERT INTO table VALUES (...)` adds rows. You can list several rows in one statement — text and dates in single quotes, booleans as `TRUE`/`FALSE`.

In [ ]:
run_sql("""
    INSERT INTO matters VALUES
        ('M-1001', 'Acme Corp',      'Contracts',   'Active', '2026-01-12',  18500.00, TRUE,  'R. Nguyen'),
        ('M-1002', 'Brightline LLC', 'Litigation',  'Active', '2025-11-03',  42750.50, TRUE,  'S. Patel'),
        ('M-1003', 'Cedar Holdings', 'M&A',         'Closed', '2025-06-21', 131200.00, TRUE,  'R. Nguyen'),
        ('M-1004', 'Dovetail Inc',   'Employment',  'Active', '2026-02-15',   9800.00, FALSE, 'T. Alvarez')
""")
run_sql("SELECT matter_id, client, practice_area, amount_billed FROM matters ORDER BY amount_billed DESC")

**What just happened:** four matters now live in the table. Everything you learned in Week 6 works on it immediately — try the `GROUP BY` below.

In [ ]:
# your Week-3 skills, on a table you built yourself
run_sql("""
    SELECT status, COUNT(*) AS matters, ROUND(SUM(amount_billed), 2) AS billed
    FROM matters
    GROUP BY status
    ORDER BY billed DESC
""")

### 5 · `CREATE TABLE AS SELECT` — a table from a query

Often you build a table *from other data*, not by typing rows. **CTAS** runs a query and stores its result as a new table — the everyday way to create curated tables (a "just the active matters" table, a summary table).

In [ ]:
run_sql("""
    CREATE OR REPLACE TABLE active_matters AS
        SELECT matter_id, client, practice_area, amount_billed
        FROM matters
        WHERE status = 'Active'
""")
run_sql("SELECT * FROM active_matters ORDER BY amount_billed DESC")

> **`Go Deeper 🔧` — `VIEW` vs `TABLE`, and `ALTER`.**
> - A **`VIEW`** is a *saved query*, not stored data — `CREATE VIEW active_v AS SELECT ...` always reflects the latest rows. A **table** is a physical copy. Use a view for a live window, a table (CTAS) to freeze/curate.
> - **`ALTER TABLE matters ADD COLUMN risk_score FLOAT`** changes a table's shape after the fact.

In [ ]:
run_sql("CREATE OR REPLACE VIEW active_view AS SELECT * FROM matters WHERE status = 'Active'")
run_sql("ALTER TABLE matters ADD COLUMN risk_score FLOAT")
run_sql("SELECT matter_id, status, risk_score FROM matters LIMIT 4")

> **`Common pitfalls ⚠️`**
>
> - **DDL vs DML:** `CREATE`/`ALTER`/`DROP` change *structure* (DDL); `INSERT`/`UPDATE`/`DELETE` change *rows* (DML).
> - **Money → `DECIMAL`/`NUMBER`, not `FLOAT`.** Floats can't represent every decimal exactly.
> - **Identifiers:** unquoted names are stored **UPPERCASE** in Snowflake (`matters` → `MATTERS`). Be consistent; avoid quoting unless you must.
> - **`CREATE OR REPLACE` drops the old table first** — great in a lesson, dangerous on real data. In production you'd think twice.

### ✍️ Your turn

In [ ]:
# TODO 1: CREATE OR REPLACE TABLE clients (client STRING, industry STRING, region STRING)

# TODO 2: INSERT two client rows of your choosing

# TODO 3: CREATE TABLE AS SELECT the high-value matters (amount_billed > 40000) into `big_matters`


<details><summary>✅ Show solution</summary>

```python
# 1
run_sql("""
    CREATE OR REPLACE TABLE clients (
        client   STRING,
        industry STRING,
        region   STRING
    )
""")

# 2
run_sql("""
    INSERT INTO clients VALUES
        ('Acme Corp',      'Manufacturing', 'Central'),
        ('Brightline LLC', 'Technology',    'North')
""")
run_sql("SELECT * FROM clients")

# 3
run_sql("""
    CREATE OR REPLACE TABLE big_matters AS
        SELECT * FROM matters WHERE amount_billed > 40000
""")
run_sql("SELECT matter_id, client, amount_billed FROM big_matters ORDER BY amount_billed DESC")
```
</details>

### 🚀 Build the artifact — the `matters` store, from scratch

You now have the capstone's storage layer: a typed `matters` table you created and populated, plus a curated `active_matters` table built from a query. This is where every later step — Claude's summaries, the API's reads — will point.

In [ ]:
summary = run_sql("""
    SELECT practice_area,
           COUNT(*)                     AS matters,
           ROUND(SUM(amount_billed), 2) AS total_billed
    FROM matters
    GROUP BY practice_area
    ORDER BY total_billed DESC
""")
print("📦 Shipped: a matters table you built, created + populated + queryable.")
summary

> **🔗 Your world.** This *is* the *Matter Intelligence* database. In Week 7 you'll **load** a real export into it (Day 2), run **analytical** queries over it (Day 3), and have Snowflake's **Cortex** summarize it in place (Day 4) — then later lessons serve it through an API. It all writes to the table you just defined.

### 📝 Recap — what you shipped

- Snowflake nests **account → warehouse (compute) → database → schema → table**; a warehouse is *compute*, billed while it runs.
- Columns have **types**: `STRING`, `DECIMAL`/`NUMBER` (money), `DATE`, `BOOLEAN`.
- **`CREATE TABLE`** defines shape; **`INSERT`** adds rows; **`CREATE TABLE AS SELECT`** builds a table from a query.
- **DDL** (`CREATE`/`ALTER`) changes structure; **DML** (`INSERT`) changes data; a **view** is a saved query, not stored rows.
- **Artifact:** your own `matters` table — the capstone's storage layer.

### 🧠 Check your understanding

1. In Snowflake, what's the difference between a **warehouse** and a **database**?
2. Which type should hold `amount_billed`, and why not `FLOAT`?
3. What does `CREATE TABLE AS SELECT` do that plain `CREATE TABLE` doesn't?
4. Is `INSERT` DDL or DML? What about `ALTER TABLE`?

<details><summary>✅ Answers</summary>

1. A **warehouse** is *compute* (the engine that runs queries, billed per-second); a **database** is *storage* (a container of schemas/tables). Snowflake separates the two.
2. **`DECIMAL`/`NUMBER`** — it stores money exactly. `FLOAT` is approximate and can introduce rounding errors on decimals.
3. CTAS **populates** the new table with a query's results in one step; plain `CREATE TABLE` makes an **empty** table you must `INSERT` into.
4. `INSERT` is **DML** (changes rows); `ALTER TABLE` is **DDL** (changes structure).
</details>

### ➡️ Next up — Week 7, Day 2: loading data at scale

Typing `INSERT` rows by hand doesn't scale to a 200,000-row export. Next lesson: Snowflake **stages** and **`COPY INTO`** — the bulk-load path — and the local equivalent, so you can load `coffee_orders.csv` (and a matters export) into a table in one shot.

*Same toolkit, no install needed.*

### 📖 Reference & glossary

| Term | Plain meaning | Legal analogy |
|---|---|---|
| **Account** | your whole Snowflake instance | the firm's building |
| **Warehouse** | compute engine (billed while running) | the associates doing the work |
| **Database** | container of schemas/tables | a filing cabinet |
| **Schema** | folder of related tables | a drawer |
| **Table** | rows + typed columns | a folder of like documents |
| **`CREATE TABLE`** | define a table's shape | design a form |
| **`INSERT`** | add rows | file a document |
| **CTAS** | table built from a query | a compiled report |
| **View** | a saved query (live, not stored) | a standing search |
| **DDL / DML** | change structure / change rows | — |

**Docs:** Snowflake data types — https://docs.snowflake.com/en/sql-reference/data-types · CREATE TABLE — https://docs.snowflake.com/en/sql-reference/sql/create-table

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*